# 07 — QLoRA Fine-Tuning

Fine-tune **Qwen3-VL-8B-Instruct** on our schema.org training dataset using QLoRA.

**Run this notebook on RunPod** — not locally. Requirements:
- GPU: 1× A100 80GB (SXM preferred; PCIe works)
- Template: RunPod PyTorch 2.1+ (CUDA 12.1)
- Setup: run `bash /workspace/schema/scripts/setup_runpod.sh` first

**Training details**:
- Base model: Qwen3-VL-8B-Instruct (quantized 4-bit NF4, ~5GB VRAM)
- LoRA adapters: r=64, alpha=128, applied to attention + MLP layers
- Effective batch size: 16 (2 per device × 8 gradient accumulation steps)
- Max sequence length: 16,384 tokens (covers 1K image + 8K HTML + output safely)
- Duration: ~3-6 hours on A100 80GB for 3K examples
- Cost: ~$8-16 at RunPod Secure rates

In [ ]:
import sys
import os
import torch

# Must be set before any CUDA init to defragment memory allocator.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Compatibility shim: set_submodule added in PyTorch 2.1 but missing on some builds.
# bitsandbytes/transformers integration calls it during 4-bit quantization setup.
if not hasattr(torch.nn.Module, 'set_submodule'):
    def _set_submodule(self, target, module):
        parent, _, last = target.rpartition('.')
        parent_mod = self.get_submodule(parent) if parent else self
        setattr(parent_mod, last, module)
    torch.nn.Module.set_submodule = _set_submodule
from pathlib import Path
from dotenv import load_dotenv

# ── RunPod working directory ────────────────────────────────────────────────
# Screenshot paths in the JSONL are relative (e.g. data/screenshots/foo.png).
# Setting cwd to the project root ensures they resolve correctly.
PROJECT_ROOT = Path('/workspace/schema')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / '.env', override=True)

# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

DATA_DIR   = PROJECT_ROOT / 'data'          # train.jsonl, eval.jsonl, screenshots/
MODELS_DIR = Path('/workspace/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Working dir: {os.getcwd()}')

## Load Config and Data

In [ ]:
import yaml

with open(PROJECT_ROOT / 'configs/training_config.yaml') as f:
    config = yaml.safe_load(f)

MODEL_ID  = config['model']['name']
HF_TOKEN  = os.getenv('HF_TOKEN')
OUTPUT_DIR = MODELS_DIR / config['training']['output_dir'].split('/')[-1]

print(f'Base model: {MODEL_ID}')
print(f'Output dir: {OUTPUT_DIR}')

# Training data lives in /workspace/schema/data/ (downloaded from HF Hub by setup script)
train_path = DATA_DIR / 'train.jsonl'
eval_path  = DATA_DIR / 'eval.jsonl'
assert train_path.exists(), f'Missing training data at {train_path}\nRun setup_runpod.sh first!'
assert eval_path.exists(),  f'Missing eval data at {eval_path}'

n_train = sum(1 for _ in open(train_path))
n_eval  = sum(1 for _ in open(eval_path))
print(f'Training examples: {n_train}')
print(f'Eval examples:     {n_eval}')

## Load Model with 4-bit Quantization

In [ ]:
from transformers import (
    AutoProcessor,
    Qwen3VLForConditionalGeneration,
    BitsAndBytesConfig,
)
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# SDPA = PyTorch 2.0+ native memory-efficient attention (no custom build needed).
# flash_attention_2 is faster but requires a compiled binary (broken on RunPod).
# eager materialises [32 heads × seq² × 4 bytes] → 72 GiB OOM → never use.
try:
    import flash_attn  # noqa: F401
    attn_impl = 'flash_attention_2'
    print('Attention: flash_attention_2')
except ImportError:
    attn_impl = 'sdpa'
    print('Attention: sdpa (memory-efficient, confirmed working for Qwen3-VL)')

print(f'Loading {MODEL_ID} with 4-bit quantization...')
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    attn_implementation=attn_impl,
    device_map='auto',
    token=HF_TOKEN,
    cache_dir=str(MODELS_DIR),
)

# max_pixels caps visual token count: 640*28*28 = 501,760 px → ~640 visual tokens max.
# Without this, a tall screenshot can generate 2000+ visual tokens and spike VRAM.
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    cache_dir=str(MODELS_DIR),
    min_pixels=256 * 28 * 28,
    max_pixels=640 * 28 * 28,
)

print(f'✓ Model ready. Parameters: {model.num_parameters()/1e9:.2f}B')
print(f'  Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M')
if torch.cuda.is_available():
    print(f'  VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## Configure LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for training (handles gradient checkpointing with 4-bit)
model = prepare_model_for_kbit_training(model)

lora_cfg = config['lora']
lora_config = LoraConfig(
    r=lora_cfg['r'],
    lora_alpha=lora_cfg['alpha'],
    lora_dropout=lora_cfg['dropout'],
    bias=lora_cfg['bias'],
    task_type=lora_cfg['task_type'],
    target_modules=lora_cfg['target_modules'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Prepare Dataset

In [ ]:
import json, torch
from pathlib import Path
from torch.utils.data import Dataset
from qwen_vl_utils import process_vision_info
from tqdm.auto import tqdm

CACHE_DIR = PROJECT_ROOT / 'data' / 'preprocessed'
CACHE_DIR.mkdir(exist_ok=True)

# ── Label masking helper ─────────────────────────────────────────────────────
_ASST_IDS = processor.tokenizer.encode('<|im_start|>assistant\n', add_special_tokens=False)

def _find_response_start(ids: list) -> int:
    n = len(_ASST_IDS)
    for i in range(len(ids) - n, -1, -1):
        if ids[i:i + n] == _ASST_IDS:
            return i + n
    return 0

# ── Preprocess & cache ────────────────────────────────────────────────────────
def preprocess_split(jsonl_path, split):
    lines = open(jsonl_path).readlines()
    skipped = 0
    for i, line in enumerate(tqdm(lines, desc=f'Preprocessing {split}')):
        cache_path = CACHE_DIR / f'{split}_{i:05d}.pt'
        if cache_path.exists():
            skipped += 1
            continue
        try:
            ex       = json.loads(line)
            messages = ex['messages']
            text     = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
            img_inputs, _ = process_vision_info(messages)
            inputs   = processor(
                text=[text],
                images=img_inputs or None,
                padding=False,
                truncation=True,
                max_length=config['training']['max_seq_length'],
                return_tensors='pt',
            )
            labels = inputs['input_ids'][0].clone()
            resp_start = _find_response_start(labels.tolist())
            labels[:resp_start] = -100
            labels[inputs['input_ids'][0] == processor.tokenizer.pad_token_id] = -100

            torch.save({
                'input_ids':      inputs['input_ids'][0],
                'attention_mask': inputs['attention_mask'][0],
                'pixel_values':   inputs['pixel_values'].to(torch.bfloat16),  # half storage vs float32
                'image_grid_thw': inputs['image_grid_thw'],
                'labels':         labels,
            }, cache_path)
        except Exception as e:
            print(f'  [skip {split}_{i}] {e}')
    print(f'{split}: {len(lines) - skipped} processed, {skipped} cached')

preprocess_split(train_path, 'train')
preprocess_split(eval_path,  'eval')

n_train = len(list(CACHE_DIR.glob('train_*.pt')))
n_eval  = len(list(CACHE_DIR.glob('eval_*.pt')))
print(f'\nCache ready — train: {n_train}  eval: {n_eval}')

# ── Dataset & collator ───────────────────────────────────────────────────────
class PreprocessedDataset(Dataset):
    def __init__(self, cache_dir, split):
        self.files = sorted(Path(cache_dir).glob(f'{split}_*.pt'))
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        return torch.load(self.files[i], map_location='cpu', weights_only=True)

train_dataset = PreprocessedDataset(CACHE_DIR, 'train')
eval_dataset  = PreprocessedDataset(CACHE_DIR, 'eval')

# No truncation — full sequences passed to model.
# SDPA handles attention memory in O(n), not O(n²), so 16K tokens is fine.
# Truncating input_ids without updating image_grid_thw causes a mismatch
# in Qwen3-VL's visual token merge step, producing a corrupt 55 GB tensor.

def collate_fn(examples):
    """Stack pre-tokenised tensors, pad to longest in batch. No truncation."""
    max_len = max(ex['input_ids'].shape[0] for ex in examples)
    input_ids = torch.stack([
        torch.nn.functional.pad(ex['input_ids'], (0, max_len - ex['input_ids'].shape[0]),
                                value=processor.tokenizer.pad_token_id)
        for ex in examples])
    attention_mask = torch.stack([
        torch.nn.functional.pad(ex['attention_mask'], (0, max_len - ex['attention_mask'].shape[0]))
        for ex in examples])
    labels = torch.stack([
        torch.nn.functional.pad(ex['labels'], (0, max_len - ex['labels'].shape[0]), value=-100)
        for ex in examples])
    pixel_values   = torch.cat([ex['pixel_values']   for ex in examples])
    image_grid_thw = torch.cat([ex['image_grid_thw'] for ex in examples])
    return {'input_ids': input_ids, 'attention_mask': attention_mask,
            'labels': labels, 'pixel_values': pixel_values, 'image_grid_thw': image_grid_thw}

print('Dataset and collate_fn ready.')

## Training

In [ ]:
from transformers import TrainingArguments, Trainer

train_cfg = config['training']

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=train_cfg['epochs'],
    per_device_train_batch_size=train_cfg['per_device_train_batch_size'],
    per_device_eval_batch_size=1,  # eval batch=8 default → [8,16K,152K] logits = 40GB OOM; force 1
    gradient_accumulation_steps=train_cfg['gradient_accumulation_steps'],
    learning_rate=train_cfg['learning_rate'],
    lr_scheduler_type=train_cfg['lr_scheduler_type'],
    warmup_ratio=train_cfg['warmup_ratio'],
    bf16=train_cfg['bf16'],
    gradient_checkpointing=train_cfg['gradient_checkpointing'],
    logging_steps=train_cfg['logging_steps'],
    save_steps=train_cfg['save_steps'],
    eval_steps=train_cfg['eval_steps'],
    eval_strategy='steps',
    save_total_limit=train_cfg['save_total_limit'],
    load_best_model_at_end=train_cfg['load_best_model_at_end'],
    metric_for_best_model=train_cfg['metric_for_best_model'],
    report_to=train_cfg.get('report_to', 'none'),
    dataloader_num_workers=0,    # 0 = main process; avoids pickling processor in workers
    gradient_checkpointing_kwargs={'use_reentrant': False},  # fixes NaN loss from broken grad flow
    remove_unused_columns=False,
)

print('Training configuration:')
print(f'  Effective batch size: {train_cfg["per_device_train_batch_size"] * train_cfg["gradient_accumulation_steps"]}')
print(f'  Learning rate: {train_cfg["learning_rate"]}')
print(f'  Epochs: {train_cfg["epochs"]}')


In [ ]:
import gc
import torch
from transformers import Trainer


class SafeTrainer(Trainer):
    """Trainer that skips OOM batches in both training and eval without crashing.

    Root-cause history:
    - Training OOM: caught RuntimeError (covers torch.OutOfMemoryError which IS a RuntimeError)
    - Eval OOM: was uncaught because OOM callback only wrapped training_step.
      eval_steps=20 triggers eval at global_step=0 (0%20==0); default batch_size=8
      → [8, 16384, 152064] logits = 40 GB → OOM.  Fixed by per_device_eval_batch_size=1
      AND this prediction_step guard as a belt-and-suspenders fallback.
    """

    def training_step(self, model, inputs, num_items_in_batch=None):
        try:
            loss = super().training_step(model, inputs, num_items_in_batch)
            if torch.isnan(loss) or torch.isinf(loss):
                print(f'[NaN/Inf] step={self.state.global_step} — skipped')
                model.zero_grad()
                return torch.tensor(0.0, device='cuda')
            return loss
        except RuntimeError as e:
            if 'out of memory' not in str(e).lower():
                raise
            torch.cuda.synchronize()   # flush pending CUDA ops
            torch.cuda.empty_cache()
            gc.collect()
            model.zero_grad()          # discard any partial gradients
            print(f'[OOM-train] step={self.state.global_step} — skipped batch')
            return torch.tensor(0.0, device='cuda')

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        """Belt-and-suspenders eval OOM guard (primary fix: per_device_eval_batch_size=1)."""
        try:
            return super().prediction_step(model, inputs, prediction_loss_only, ignore_keys)
        except RuntimeError as e:
            if 'out of memory' not in str(e).lower():
                raise
            torch.cuda.empty_cache()
            gc.collect()
            print(f'[OOM-eval] step={self.state.global_step} — skipped eval sample')
            return (None, None, None)


# Create trainer (training_args, datasets, collate_fn, model all defined above)
trainer = SafeTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_fn,
)

print(f'Trainer ready. Steps per epoch: {len(train_dataset) // training_args.gradient_accumulation_steps}')
print('Starting fine-tuning...')
trainer.train()

trainer.save_model(str(OUTPUT_DIR / 'final'))
processor.save_pretrained(str(OUTPUT_DIR / 'final'))
print(f'\nTraining complete. Model saved to {OUTPUT_DIR / "final"}')


In [ ]:
# Training loss curve
import matplotlib.pyplot as plt
import pandas as pd

log_history = trainer.state.log_history
train_logs = [l for l in log_history if 'loss' in l and 'eval_loss' not in l]
eval_logs = [l for l in log_history if 'eval_loss' in l]

if train_logs:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot([l['step'] for l in train_logs], [l['loss'] for l in train_logs], label='Train loss')
    if eval_logs:
        ax.plot([l['step'] for l in eval_logs], [l['eval_loss'] for l in eval_logs], label='Eval loss')
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.set_title('Fine-tuning Loss')
    ax.legend()
    plt.tight_layout()
    plt.show()

print('Next step: 08_evaluation.ipynb')